# Demo E — Pseudo-labels: self-training and co-training

**Class 11 · E1–E2** (slides: `slides/11_autoencoders.md`)

* **E1:** self-training. A logistic regression is fitted on 50 labelled digits (5 per class); unlabelled digits it is sure about (top probability at least $\tau$) receive a pseudo-label and join the training set. Sweep of $\tau$, the history of the rounds, the twin `SelfTrainingClassifier` and a failure case: wrong labels.
* **E2:** co-training: two views of every digit (top and bottom halves of the image, or a random split of the pixels), two classifiers that teach each other, compared with self-training and with the two views without unlabelled data.

In [ ]:
import numpy as np
import polars as pl
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import SelfTrainingClassifier
from threadpoolctl import threadpool_limits

from faa import demo, plotlib, semisup
from faa.demo import show, slide

demo.setup("11")
threadpool_limits(1)  # small matrices: one BLAS thread is faster than several
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)

## Slide: Self-Training in Rounds (E1)

Same split as Demo D (70 / 30, 1257 train rows). 5 labelled digits per class (50 labels), the other 1207 train rows are unlabelled; accuracy on the 540 test rows.
Threshold $\tau = 0.9$, at most 10 rounds.

In [ ]:
X, y = semisup.digits_flat()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
lab = semisup.few_labels(ytr, 5, 0)
unl = np.setdiff1d(np.arange(len(Xtr)), lab)
base = LogisticRegression(C=10, max_iter=3000).fit(Xtr[lab], ytr[lab]).score(Xte, yte)
model, hist = semisup.self_train(Xtr[lab], ytr[lab], Xtr[unl], 0.9, 10, y_true_u=ytr[unl])
slide("Self-Training in Rounds")
print(f"{len(lab)} labelled, {len(unl)} unlabelled; labelled only: test accuracy {base:.3f}")
show(pl.DataFrame(hist))
print(f"after the rounds: test accuracy {model.score(Xte, yte):.3f}")

fig, ax = plotlib.fig("col", 1.6)
accs = [base]
for r in range(1, len(hist) + 1):
    m, _ = semisup.self_train(Xtr[lab], ytr[lab], Xtr[unl], 0.9, r)
    accs.append(m.score(Xte, yte))
ax.plot(range(len(accs)), accs, color=BLUE, marker="o", ms=3)
ax.set_xlabel("round")
ax.set_ylabel("test accuracy")
plotlib.save(fig, "e1_rounds")

## Slide: The Threshold (E1)

5 draws of the 50 labels for every threshold $\tau$; the last column is the share of unlabelled rows that were pseudo-labelled and the accuracy of those pseudo-labels.

In [ ]:
rows = []
for tau in (0.5, 0.7, 0.8, 0.9, 0.95, 0.99):
    accs, shares, pacc = [], [], []
    for seed in range(5):
        lb = semisup.few_labels(ytr, 5, seed)
        u = np.setdiff1d(np.arange(len(Xtr)), lb)
        m, h = semisup.self_train(Xtr[lb], ytr[lb], Xtr[u], tau, 10, y_true_u=ytr[u])
        accs.append(m.score(Xte, yte))
        shares.append(sum(r["added"] for r in h) / len(u))
        pacc.append(np.nansum([r["pseudo_acc"] * r["added"] for r in h]) / max(1, sum(r["added"] for r in h)))
    base_s = np.mean(
        [
            LogisticRegression(C=10, max_iter=3000)
            .fit(Xtr[semisup.few_labels(ytr, 5, s)], ytr[semisup.few_labels(ytr, 5, s)])
            .score(Xte, yte)
            for s in range(5)
        ]
    )
    rows.append(
        {
            "tau": tau,
            "acc": np.mean(accs),
            "std": np.std(accs),
            "labelled_only": base_s,
            "pseudo_share": np.mean(shares),
            "pseudo_acc": np.mean(pacc),
        }
    )
sweep = pl.DataFrame(rows)
slide("The Threshold")
show(sweep)

fig, ax = plotlib.fig("col", 1.6)
ax.errorbar(sweep["tau"], sweep["acc"], sweep["std"], color=BLUE, marker="o", ms=3, capsize=2, label="self-training")
ax.axhline(sweep["labelled_only"][0], color=GREY, ls="--", lw=0.8, label="labelled only")
ax.set_xlabel(r"threshold $\tau$")
ax.set_ylabel("test accuracy")
ax.legend(loc="lower right")
plotlib.save(fig, "e1_threshold")

## Twin: `SelfTrainingClassifier` (E1)

scikit-learn's version on the same rows (unlabelled rows get the label $-1$), same $\tau = 0.9$.

In [ ]:
y_semi = ytr.copy()
y_semi[unl] = -1
sk = SelfTrainingClassifier(LogisticRegression(C=10, max_iter=3000), threshold=0.9, max_iter=10).fit(Xtr, y_semi)
agree = float(np.mean(sk.predict(Xte) == model.predict(Xte)))
slide("Self-Training in Rounds (twin)")
print(
    f"scikit-learn SelfTrainingClassifier: test accuracy {sk.score(Xte, yte):.3f}; ours {model.score(Xte, yte):.3f}; agreement on the test rows {agree:.3f}"
)

## Slide: Confirmation Bias (E1)

A fraction of the 50 labels is wrong (flipped to another class). The pseudo-labels inherit the errors, and the model grows more sure of them.

In [ ]:
rows = []
for frac in (0.0, 0.1, 0.2, 0.3):
    a0, a1 = [], []
    for seed in range(5):
        lb = semisup.few_labels(ytr, 5, seed)
        yl = ytr[lb].copy()
        yl = np.where(
            np.isin(
                np.arange(len(lb)), np.random.default_rng(seed).choice(len(lb), int(frac * len(lb)), replace=False)
            ),
            (yl + np.random.default_rng(seed).integers(1, 10, len(lb))) % 10,
            yl,
        )
        u = np.setdiff1d(np.arange(len(Xtr)), lb)
        a0.append(LogisticRegression(C=10, max_iter=3000).fit(Xtr[lb], yl).score(Xte, yte))
        m, _ = semisup.self_train(Xtr[lb], yl, Xtr[u], 0.9, 10)
        a1.append(m.score(Xte, yte))
    rows.append(
        {
            "wrong_labels": frac,
            "labelled_only": np.mean(a0),
            "self_training": np.mean(a1),
            "gain": np.mean(a1) - np.mean(a0),
        }
    )
bias = pl.DataFrame(rows)
slide("Confirmation Bias")
show(bias)

fig, ax = plotlib.fig("col", 1.6)
ax.plot(bias["wrong_labels"], bias["labelled_only"], color=GREY, marker="o", ms=3, label="labelled only")
ax.plot(bias["wrong_labels"], bias["self_training"], color=RED, marker="o", ms=3, label="self-training")
ax.set_xlabel("share of wrong labels")
ax.set_ylabel("test accuracy")
ax.legend(loc="lower left")
plotlib.save(fig, "e1_bias")

## Slide: Co-Training (E2)

Two views of every digit: the **top four rows** of the image (32 pixels) and the **bottom four rows** (32 pixels). Classifier A reads the top, classifier B the bottom. Each round, A hands the unlabelled rows it is sure about ($\tau = 0.9$) to B with its pseudo-label, and B does the same for A.
The prediction is the average of the two probabilities. 5 draws of 50 labels, accuracy on the 540 test rows. Co-training works when the views are conditionally independent given the class; two halves of one digit are not.

In [ ]:
from sklearn.linear_model import LogisticRegression as LR

top, bot = slice(0, 32), slice(32, 64)
rng_views = np.random.default_rng(0)
perm = rng_views.permutation(64)
views = {"top / bottom halves": (top, bot), "random pixel split": (np.sort(perm[:32]), np.sort(perm[32:]))}
rows = []
for name, (va, vb) in views.items():
    res = {k: [] for k in ("both views, labelled only", "self-training (both views)", "co-training")}
    pa = []
    for seed in range(5):
        lb = semisup.few_labels(ytr, 5, seed)
        u = np.setdiff1d(np.arange(len(Xtr)), lb)
        ma0 = LR(C=10, max_iter=3000).fit(Xtr[lb][:, va], ytr[lb])
        mb0 = LR(C=10, max_iter=3000).fit(Xtr[lb][:, vb], ytr[lb])
        p0 = ma0.predict_proba(Xte[:, va]) + mb0.predict_proba(Xte[:, vb])
        res["both views, labelled only"].append(float(np.mean(ma0.classes_[p0.argmax(1)] == yte)))
        st, _ = semisup.self_train(Xtr[lb], ytr[lb], Xtr[u], 0.9, 10)
        res["self-training (both views)"].append(st.score(Xte, yte))
        ma, mb, h = semisup.co_train(
            Xtr[lb][:, va], Xtr[lb][:, vb], ytr[lb], Xtr[u][:, va], Xtr[u][:, vb], 0.9, 10, y_true_u=ytr[u]
        )
        p1 = ma.predict_proba(Xte[:, va]) + mb.predict_proba(Xte[:, vb])
        res["co-training"].append(float(np.mean(ma.classes_[p1.argmax(1)] == yte)))
        pa.append(np.nanmean([r["pseudo_acc"] for r in h]))
    for k, v in res.items():
        rows.append({"views": name, "method": k, "acc": np.mean(v), "std": np.std(v)})
    rows.append(
        {"views": name, "method": "pseudo-label accuracy (rounds)", "acc": float(np.mean(pa)), "std": float(np.std(pa))}
    )
co = pl.DataFrame(rows)
slide("Co-Training")
show(co)

fig, ax = plotlib.fig("col", 1.7)
sub = co.filter(pl.col("method") != "pseudo-label accuracy (rounds)")
labs = ["labelled\\nonly", "self-\\ntraining", "co-\\ntraining"]
w = 0.38
for j, (name, c) in enumerate(zip(views, (BLUE, ORANGE), strict=True)):
    s2 = sub.filter(pl.col("views") == name)
    ax.bar(np.arange(3) + (j - 0.5) * w, s2["acc"], w, yerr=s2["std"], color=c, capsize=2, label=name)
ax.set_xticks(range(3), labs)
ax.set_ylim(0.6, 0.95)
ax.set_ylabel("test accuracy")
ax.legend(loc="upper left", fontsize=5.5)
plotlib.save(fig, "e2_cotrain")

## Sweep: does a stricter threshold rescue co-training? (E2)

Random pixel split, 5 draws of the 50 labels, $\tau$ from 0.9 to 0.999 and 3 or 10 rounds.

In [ ]:
va, vb = views["random pixel split"]
rows = []
for tau in (0.9, 0.99, 0.999):
    for rounds in (3, 10):
        accs = []
        for seed in range(5):
            lb = semisup.few_labels(ytr, 5, seed)
            u = np.setdiff1d(np.arange(len(Xtr)), lb)
            ma, mb, _ = semisup.co_train(
                Xtr[lb][:, va], Xtr[lb][:, vb], ytr[lb], Xtr[u][:, va], Xtr[u][:, vb], tau, rounds
            )
            p1 = ma.predict_proba(Xte[:, va]) + mb.predict_proba(Xte[:, vb])
            accs.append(float(np.mean(ma.classes_[p1.argmax(1)] == yte)))
        rows.append({"tau": tau, "rounds": rounds, "acc": np.mean(accs), "std": np.std(accs)})
slide("Co-Training (sweep)")
show(pl.DataFrame(rows))